In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPool2D
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import TensorBoard
import splitfolders

#NAME="JugendForschtTest1"
#tensorboard = TensorBoard(log_dir = 'logs/'.format(NAME))

splitfolders.ratio(r"C:\Users\Luis\Desktop\Back", output= r"C:\Users\Luis\anaconda3\Backoutput",
    seed=1337, ratio=(.7, .2, .1), group_prefix=None, move=False)



train_path =r"C:\Users\Luis\anaconda3\Backoutput\train"
validation_path =r"C:\Users\Luis\anaconda3\Backoutput\val"
test_path = r"C:\Users\Luis\anaconda3\Backoutput\test"



model = Sequential()
model.add(Conv2D(32,kernel_size=(3,3),input_shape=(200,200,3),activation="relu"))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(32,kernel_size=(3,3),activation="relu"))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(16,kernel_size=(3,3),activation="relu"))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Flatten())
model.add(Dense(1024, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(15,activation="softmax"))

model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["accuracy"])
batch_size= 32

train_datagen = ImageDataGenerator(rescale=1./255,
                   shear_range=0.7,
                   horizontal_flip=True,
                   zoom_range=0.7)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_Generator = train_datagen.flow_from_directory(train_path,
                                                          target_size=[200,200],
                                                          batch_size= batch_size,
                                                          color_mode="rgb",
                                                          class_mode="categorical",
                                                          shuffle=True)
val_Generator = val_datagen.flow_from_directory(validation_path,
                                                  target_size=[200,200],
                                                    batch_size=batch_size,
                                                    color_mode="rgb",
                                                    class_mode="categorical",
                                                    shuffle=False)
test_Generator = test_datagen.flow_from_directory(test_path,
                                                  target_size=[200,200],
                                                    batch_size=batch_size,
                                                    color_mode="rgb",
                                                    class_mode="categorical")

history = model.fit(train_Generator, epochs=15, validation_data=val_Generator)

Copying files: 3000 files [00:02, 1013.95 files/s]


Found 2100 images belonging to 15 classes.
Found 600 images belonging to 15 classes.
Found 300 images belonging to 15 classes.
Epoch 1/15
66/66 [==============================] - 36s 543ms/step - loss: 1.6221 - accuracy: 0.4652 - val_loss: 0.4017 - val_accuracy: 0.9067
Epoch 2/15
66/66 [==============================] - 35s 535ms/step - loss: 0.5601 - accuracy: 0.8171 - val_loss: 0.2043 - val_accuracy: 0.9467
Epoch 3/15
66/66 [==============================] - 35s 525ms/step - loss: 0.3816 - accuracy: 0.8790 - val_loss: 0.1392 - val_accuracy: 0.9667
Epoch 4/15
66/66 [==============================] - 35s 525ms/step - loss: 0.3008 - accuracy: 0.8990 - val_loss: 0.1407 - val_accuracy: 0.9383
Epoch 5/15
66/66 [==============================] - 35s 533ms/step - loss: 0.2260 - accuracy: 0.9267 - val_loss: 0.0747 - val_accuracy: 0.9767
Epoch 6/15
66/66 [==============================] - 34s 510ms/step - loss: 0.1758 - accuracy: 0.9500 - val_loss: 0.0642 - val_accuracy: 0.9833
Epoch 7/15
66/6

In [11]:
model.evaluate(test_Generator)

10/10 [==============================] - 1s 94ms/step - loss: 0.0242 - accuracy: 0.9933


[0.0242092777043581, 0.9933333396911621]

In [3]:
model.save("lasttest.keras")

In [ ]:
tensorboard --logdir logs/test

In [12]:
# Convert the model.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_LATENCY]
tflite_model = converter.convert()

# Save the model.
with open('lasttest.tflite', 'wb') as f:
  f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\Luis\AppData\Local\Temp\tmpknrhrfzs\assets


INFO:tensorflow:Assets written to: C:\Users\Luis\AppData\Local\Temp\tmpknrhrfzs\assets


In [23]:
from tflite_support.metadata_writers import image_classifier
from tflite_support.metadata_writers import writer_utils

ImageClassifierWriter = image_classifier.MetadataWriter
_MODEL_PATH = r"C:\Users\Luis\anaconda3\CroppedJF.tflite"
# Task Library expects label files that are in the same format as the one below.
_LABEL_FILE = r"C:\Users\Luis\anaconda3\CroppedJFlabel.txt"
_SAVE_TO_PATH = r"C:\Users\Luis\anaconda3\CroppedJF_with_meta.tflite"
# Normalization parameters is required when reprocessing the image. It is
# optional if the image pixel values are in range of [0, 255] and the input
# tensor is quantized to uint8. See the introduction for normalization and
# quantization parameters below for more details.
# https://www.tensorflow.org/lite/models/convert/metadata#normalization_and_quantization_parameters)
_INPUT_NORM_MEAN = 0.5
_INPUT_NORM_STD = 0.5

# Create the metadata writer.
writer = ImageClassifierWriter.create_for_inference(
    writer_utils.load_file(_MODEL_PATH), [_INPUT_NORM_MEAN], [_INPUT_NORM_STD],
    [_LABEL_FILE])

# Verify the metadata generated by metadata writer.
print(writer.get_metadata_json())

# Populate the metadata into the model.
writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

{
  "name": "ImageClassifier",
  "description": "Identify the most prominent object in the image from a known set of categories.",
  "subgraph_metadata": [
    {
      "input_tensor_metadata": [
        {
          "name": "image",
          "description": "Input image to be classified.",
          "content": {
            "content_properties_type": "ImageProperties",
            "content_properties": {
              "color_space": "RGB"
            }
          },
          "process_units": [
            {
              "options_type": "NormalizationOptions",
              "options": {
                "mean": [
                  0.5
                ],
                "std": [
                  0.5
                ]
              }
            }
          ],
          "stats": {
            "max": [
              509.0
            ],
            "min": [
              -1.0
            ]
          }
        }
      ],
      "output_tensor_metadata": [
        {
          "name": "probabi

In [28]:
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model

# q_aware stands for for quantization aware.
q_aware_model = quantize_model(model)

# `quantize_model` requires a recompile.
q_aware_model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

q_aware_model.summary()

ModuleNotFoundError: No module named 'tensorflow_model_optimization'